# Step 3: Musical Chord Recognition with `pipeline.py` & `btc-chord`

This notebook uses **`pipeline.py`** to perform automatic chord recognition on **`data/bass_other.wav`** using the **BTC (Bi-directional Transformer for Chord Recognition)** model (`puar-playground/btc-chord`).

### Pipeline:
1. **Load Accompaniment**: Reads `data/bass_other.wav` from `combine.ipynb`.
2. **Initialize BTC Model**: Uses `get_btc_model()` on GPU (`cuda`).
3. **Predict Chords with Visual Progress Bar**: Runs transformer inference with `predict_chords(..., show_progress=True)`.
4. **Export CSV**: Saves structured timeline to **`data/chords.csv`**.
5. **Progression Summary**: Displays top chords and sequence.

> **Kernel**: Make sure the kernel is set to **`Python (seperate)`**.

## 1. Setup & Device Configuration

In [ ]:
from pathlib import Path
import torch
import pandas as pd
from pipeline import get_btc_model, predict_chords

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Define Input & Output Paths

In [ ]:
data_dir = Path("data")
accompaniment_path = data_dir / "bass_other.wav"
chords_csv_path = data_dir / "chords.csv"

print(f"Accompaniment input: {accompaniment_path}")
print(f"Chords CSV target  : {chords_csv_path}")

assert accompaniment_path.exists(), (
    f"Missing accompaniment file: {accompaniment_path}.\n"
    "Please run separate.ipynb and combine.ipynb first!"
)
print("\n✓ Input accompaniment audio file found!")

## 3. Load BTC Chord Model via `pipeline.py`

In [ ]:
print(f"Loading BTC Chord model on {device}...")
model = get_btc_model()
print("Model ready for inference!")

## 4. Run Chord Recognition with Live Progress Bar

In [ ]:
# Run prediction with live chunk progress bar
raw_chords = predict_chords(model, accompaniment_path, show_progress=True)

# Convert to DataFrame
df_chords = pd.DataFrame(raw_chords)
if not df_chords.empty:
    df_chords["duration"] = (df_chords["end"] - df_chords["start"]).round(3)

print(f"\nExtracted {len(df_chords)} chord segments.\n")
print("First 15 chord segments:")
print(df_chords.head(15).to_string(index=False))

## 5. Save Predictions to CSV

In [ ]:
df_chords.to_csv(chords_csv_path, index=False)
print(f"✅ Successfully saved chords to: {chords_csv_path}")

## 6. Chord Summary & Progression Overview

In [ ]:
# Filter out 'N' (no-chord) to see most frequent chords
active_chords = df_chords[df_chords["chord"] != "N"]
chord_counts = active_chords.groupby("chord")["duration"].sum().sort_values(ascending=False)

print("Top Chords by Total Duration (seconds):")
for chord, dur in chord_counts.head(10).items():
    print(f"  • {chord:<10} : {dur:.2f} s")

# Simplified progression (consecutive unique chords)
progression = []
for chord in df_chords["chord"]:
    if not progression or progression[-1] != chord:
        progression.append(chord)

print(f"\nChord Progression Sequence ({len(progression)} transitions):")
print(" -> ".join(progression[:20]) + (" ..." if len(progression) > 20 else ""))